# NSGA-II for Feature Selection

**Module 06 — Evolutionary & Swarm Methods**

Estimated time: 15 minutes

## What you will build

1. A complete NSGA-II feature selector using DEAP with two objectives: accuracy and feature count
2. Pareto front visualisation
3. Knee point detection for automatic solution selection
4. Direct comparison with single-objective GA from Module 5

**References**:
- Deb, Pratap, Agarwal & Meyarivan (2002). A Fast and Elitist Multiobjective Genetic Algorithm: NSGA-II. *IEEE TEC*, 6(2), 182–197.
- DEAP documentation: https://deap.readthedocs.io

In [ ]:
# Dependencies
# pip install deap scikit-learn matplotlib numpy

import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from deap import base, creator, tools
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
print("Libraries loaded.")

## 1. Dataset

We use the breast cancer Wisconsin dataset: 569 samples, 30 features, binary classification.

In [ ]:
data = load_breast_cancer()
X_raw, y = data.data, data.target

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

N_FEATURES = X.shape[1]
FEATURE_NAMES = data.feature_names

print(f"Dataset: {X.shape[0]} samples, {N_FEATURES} features")
print(f"Classes: {np.bincount(y)} (benign={y.sum()}, malignant={(y==0).sum()})")

# Baseline: all features
clf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_scores = cross_val_score(clf_baseline, X, y, cv=cv, scoring='accuracy')
BASELINE_ERROR = 1.0 - baseline_scores.mean()
print(f"\nBaseline (all {N_FEATURES} features): {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f} accuracy")
print(f"Baseline error rate: {BASELINE_ERROR:.4f}")

## 2. NSGA-II Setup with DEAP

NSGA-II minimises two objectives simultaneously:
- **Objective 1**: Cross-validation error rate (minimise)
- **Objective 2**: Feature fraction = selected_features / total_features (minimise)

DEAP uses `weights=(-1.0, -1.0)` for minimisation (it internally maximises, so negative weights flip the sign).

In [ ]:
# Clear any existing DEAP creators to avoid re-registration errors
if 'FitnessMulti' in creator.__dict__:
    del creator.FitnessMulti
if 'Individual' in creator.__dict__:
    del creator.Individual

# Two-objective fitness: both minimised
creator.create("FitnessMulti", base.Fitness, weights=(-1.0, -1.0))
creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=N_FEATURES,
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

print("DEAP setup complete.")
print(f"Individual length: {N_FEATURES} bits")
print("Objective 1: error rate (minimise)")
print("Objective 2: feature fraction (minimise)")

## 3. Fitness Function with Caching

Cross-validation is the bottleneck. We cache evaluations by converting each individual to a tuple key.

In [ ]:
_fitness_cache = {}

def evaluate_individual(individual):
    """
    Evaluate a binary feature mask on two objectives.
    
    Returns
    -------
    tuple
        (error_rate, feature_fraction)
    """
    key = tuple(individual)
    if key in _fitness_cache:
        return _fitness_cache[key]
    
    mask = np.array(individual, dtype=bool)
    n_selected = mask.sum()
    
    if n_selected == 0:
        result = (1.0, 0.0)
        _fitness_cache[key] = result
        return result
    
    X_sub = X[:, mask]
    clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_sub, y, cv=cv, scoring='accuracy')
    
    error_rate = 1.0 - scores.mean()
    feature_fraction = n_selected / N_FEATURES
    
    result = (error_rate, feature_fraction)
    _fitness_cache[key] = result
    return result

toolbox.register("evaluate", evaluate_individual)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=1.0 / N_FEATURES)
toolbox.register("select", tools.selNSGA2)

print("Fitness function registered with caching.")

## 4. Run NSGA-II

Main NSGA-II loop:
1. Generate offspring via `selTournamentDCD` (dominance + crowding distance tournament)
2. Apply crossover and mutation
3. Evaluate new individuals
4. Select next generation from combined parent+offspring pool using `selNSGA2`

In [ ]:
POPULATION_SIZE = 80
N_GENERATIONS = 50
CXPB = 0.7   # crossover probability
MUTPB = 0.2  # mutation probability

random.seed(42)
np.random.seed(42)

# Initialise population
population = toolbox.population(n=POPULATION_SIZE)

# Evaluate initial population
fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

# Assign crowding distance to initial population (required by selTournamentDCD)
population = toolbox.select(population, len(population))

# Track Pareto front evolution
pareto_history = []

print(f"Running NSGA-II: {POPULATION_SIZE} individuals × {N_GENERATIONS} generations")
print(f"Cache hits will accelerate evaluation.\n")

for gen in range(N_GENERATIONS):
    # Step 1: generate offspring via dominance-crowding tournament
    offspring = tools.selTournamentDCD(population, len(population))
    offspring = [toolbox.clone(ind) for ind in offspring]

    # Step 2: crossover
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CXPB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # Step 3: mutation
    for mutant in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Step 4: evaluate invalid individuals
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    # Step 5: NSGA-II selection (non-dominated sort + crowding)
    population = toolbox.select(population + offspring, POPULATION_SIZE)

    # Track Pareto front
    front = tools.sortNondominated(population, len(population), first_front_only=True)[0]
    pareto_history.append(len(front))

    if gen % 10 == 0 or gen == N_GENERATIONS - 1:
        errors = [ind.fitness.values[0] for ind in front]
        print(f"Gen {gen:3d} | Pareto size={len(front):3d} | "
              f"best_error={min(errors):.4f} | "
              f"cache_size={len(_fitness_cache):4d}")

print(f"\nDone. Total unique evaluations: {len(_fitness_cache)}")

## 5. Extract and Inspect the Pareto Front

In [ ]:
# Extract final Pareto front
pareto_front = tools.sortNondominated(
    population, len(population), first_front_only=True
)[0]

# Sort by error rate (ascending) for display
pareto_front_sorted = sorted(pareto_front, key=lambda ind: ind.fitness.values[0])

print(f"Pareto front contains {len(pareto_front_sorted)} solutions\n")
print(f"{'Sol':>4} {'Error':>8} {'Features':>10} {'Accuracy':>10} {'vs Baseline':>12}")
print("-" * 50)

for i, ind in enumerate(pareto_front_sorted):
    err, feat_frac = ind.fitness.values
    n_feat = sum(ind)
    delta_vs_baseline = err - BASELINE_ERROR
    print(f"{i:>4} {err:>8.4f} {n_feat:>10} {1-err:>10.4f} {delta_vs_baseline:>+12.4f}")

## 6. Visualise the Pareto Front

In [ ]:
def find_knee_point(objectives: np.ndarray) -> int:
    """
    Find the knee point of a Pareto front via maximum perpendicular
    distance from the line connecting the extreme solutions.

    Parameters
    ----------
    objectives : ndarray of shape (n_solutions, 2)
        [error_rate, feature_fraction] for each Pareto solution.

    Returns
    -------
    int
        Index of the knee point in the sorted objectives array.
    """
    if len(objectives) <= 2:
        return 0

    # Already sorted by error (first objective)
    start = objectives[0]
    end = objectives[-1]
    line_vec = end - start
    line_len = np.linalg.norm(line_vec)

    if line_len < 1e-12:
        return 0

    # Perpendicular distance from the line for each point
    distances = []
    for pt in objectives:
        vec = pt - start
        proj_scalar = np.dot(vec, line_vec) / (line_len ** 2)
        proj_pt = start + proj_scalar * line_vec
        dist = np.linalg.norm(pt - proj_pt)
        distances.append(dist)

    return int(np.argmax(distances))


# Prepare objectives array
objectives = np.array([
    [ind.fitness.values[0], sum(ind)]
    for ind in pareto_front_sorted
])
# Normalise feature count to fraction for knee detection
objectives_norm = np.column_stack([
    objectives[:, 0],
    objectives[:, 1] / N_FEATURES
])

knee_idx = find_knee_point(objectives_norm)
knee_solution = pareto_front_sorted[knee_idx]
knee_error = knee_solution.fitness.values[0]
knee_features = sum(knee_solution)

print(f"Knee point: solution {knee_idx}")
print(f"  Error rate: {knee_error:.4f} (accuracy: {1-knee_error:.4f})")
print(f"  Features: {knee_features}/{N_FEATURES} ({knee_features/N_FEATURES:.1%})")
print(f"  vs baseline: {knee_error - BASELINE_ERROR:+.4f} error rate")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Pareto front ---
ax = axes[0]
errors = objectives[:, 0]
features = objectives[:, 1].astype(int)

ax.scatter(features, errors, c='steelblue', s=70, zorder=5, label='Pareto solutions')
ax.plot(features, errors, 'b--', alpha=0.4)

# Knee point
ax.scatter(features[knee_idx], errors[knee_idx],
           c='red', s=200, zorder=6, marker='*', label=f'Knee point ({knee_features} features)')

# Baseline
ax.axhline(BASELINE_ERROR, color='gray', linestyle=':', linewidth=1.5,
           label=f'Baseline (all {N_FEATURES} features)')

# Line connecting extremes (for knee visualisation)
ax.plot([features[0], features[-1]], [errors[0], errors[-1]],
        'k:', alpha=0.3, linewidth=1)

ax.set_xlabel('Number of features selected', fontsize=12)
ax.set_ylabel('Cross-validation error rate', fontsize=12)
ax.set_title('NSGA-II Pareto Front\nAccuracy vs. Feature Count', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Right: Pareto front evolution ---
ax2 = axes[1]
ax2.plot(pareto_history, color='darkblue', linewidth=2)
ax2.fill_between(range(len(pareto_history)), pareto_history, alpha=0.2, color='steelblue')
ax2.set_xlabel('Generation', fontsize=12)
ax2.set_ylabel('Pareto front size', fontsize=12)
ax2.set_title('Pareto Front Growth Over Generations', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nsga2_pareto_front.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved: nsga2_pareto_front.png")

## 7. Knee Point — Feature Analysis

In [ ]:
knee_mask = np.array(knee_solution, dtype=bool)
selected_feature_names = FEATURE_NAMES[knee_mask]

print(f"Knee point solution — selected features ({knee_features}/{N_FEATURES}):")
print()
for name in selected_feature_names:
    print(f"  + {name}")

# Validate on fresh CV
clf_knee = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knee_scores = cross_val_score(clf_knee, X[:, knee_mask], y, cv=cv5, scoring='accuracy')

print(f"\n5-fold CV validation:")
print(f"  Accuracy: {knee_scores.mean():.4f} ± {knee_scores.std():.4f}")
print(f"  Baseline: {1-BASELINE_ERROR:.4f}")
print(f"  Difference: {knee_scores.mean() - (1-BASELINE_ERROR):+.4f}")

## 8. Three Solution Selection Strategies

In [ ]:
def select_by_weighted_utility(
    pareto_sorted, error_weight=0.7, complexity_weight=0.3
):
    """Select solution maximising weighted utility on normalised objectives."""
    errors = np.array([ind.fitness.values[0] for ind in pareto_sorted])
    features = np.array([sum(ind) / N_FEATURES for ind in pareto_sorted])

    # Normalise to [0, 1]
    err_norm = (errors - errors.min()) / (errors.max() - errors.min() + 1e-12)
    feat_norm = (features - features.min()) / (features.max() - features.min() + 1e-12)

    # Utility: minimise both (lower is better)
    utility = error_weight * err_norm + complexity_weight * feat_norm
    return int(np.argmin(utility))


def select_by_constraint(pareto_sorted, max_features):
    """Select best-accuracy solution using at most max_features."""
    feasible = [
        (i, ind) for i, ind in enumerate(pareto_sorted) if sum(ind) <= max_features
    ]
    if not feasible:
        # Relax: pick fewest features
        i_best = min(range(len(pareto_sorted)), key=lambda i: sum(pareto_sorted[i]))
        return i_best
    # Among feasible, pick lowest error
    return min(feasible, key=lambda t: t[1].fitness.values[0])[0]


# Strategy 1: Knee point
sol_knee = pareto_front_sorted[knee_idx]

# Strategy 2: Weighted utility (70% accuracy, 30% complexity)
wu_idx = select_by_weighted_utility(pareto_front_sorted)
sol_wu = pareto_front_sorted[wu_idx]

# Strategy 3: Constraint (at most 12 features)
MAX_FEATURES = 12
con_idx = select_by_constraint(pareto_front_sorted, MAX_FEATURES)
sol_con = pareto_front_sorted[con_idx]

print("Solution selection comparison:")
print(f"{'Strategy':<22} {'Error':>8} {'Accuracy':>10} {'Features':>10}")
print("-" * 55)
print(f"{'Knee point':<22} {sol_knee.fitness.values[0]:>8.4f} "
      f"{1-sol_knee.fitness.values[0]:>10.4f} {sum(sol_knee):>10}")
print(f"{'Weighted utility':<22} {sol_wu.fitness.values[0]:>8.4f} "
      f"{1-sol_wu.fitness.values[0]:>10.4f} {sum(sol_wu):>10}")
print(f"{'Constraint (≤{MAX_FEATURES})':<22} {sol_con.fitness.values[0]:>8.4f} "
      f"{1-sol_con.fitness.values[0]:>10.4f} {sum(sol_con):>10}")
print(f"{'Baseline (all feats)':<22} {BASELINE_ERROR:>8.4f} "
      f"{1-BASELINE_ERROR:>10.4f} {N_FEATURES:>10}")

## 9. Comparison: NSGA-II vs Single-Objective GA

In [ ]:
def run_single_objective_ga(
    alpha_feature_penalty=0.1,
    population_size=80,
    n_generations=50,
    random_state=42,
):
    """
    Single-objective GA: minimise error_rate + alpha * feature_fraction.
    Returns (best_individual, best_fitness_history).
    """
    rng = random.Random(random_state)
    np_rng = np.random.default_rng(random_state)

    def scalar_fitness(individual):
        mask = np.array(individual, dtype=bool)
        if not mask.any():
            return 1.0 + alpha_feature_penalty
        X_sub = X[:, mask]
        clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        err = 1.0 - cross_val_score(clf, X_sub, y, cv=cv, scoring='accuracy').mean()
        return err + alpha_feature_penalty * (mask.sum() / N_FEATURES)

    # Initialise
    pop = [np_rng.integers(0, 2, N_FEATURES).tolist() for _ in range(population_size)]
    fit = [scalar_fitness(ind) for ind in pop]
    best_hist = [min(fit)]

    for gen in range(n_generations):
        new_pop = []
        for _ in range(population_size):
            # Tournament selection
            p1 = min(rng.sample(range(population_size), 3), key=lambda i: fit[i])
            p2 = min(rng.sample(range(population_size), 3), key=lambda i: fit[i])
            # Crossover
            cx = rng.randint(1, N_FEATURES - 1)
            child = pop[p1][:cx] + pop[p2][cx:]
            # Mutation
            child = [1 - b if rng.random() < 1.0 / N_FEATURES else b for b in child]
            if sum(child) == 0:
                child[rng.randint(0, N_FEATURES - 1)] = 1
            new_pop.append(child)

        new_fit = [scalar_fitness(ind) for ind in new_pop]

        # Elitist: keep best from old + new
        all_pop = pop + new_pop
        all_fit = fit + new_fit
        top = np.argsort(all_fit)[:population_size]
        pop = [all_pop[i] for i in top]
        fit = [all_fit[i] for i in top]
        best_hist.append(fit[0])

    best_ind = pop[0]
    return best_ind, best_hist


print("Running single-objective GA (this takes ~1-2 minutes)...")
ga_best, ga_history = run_single_objective_ga(
    alpha_feature_penalty=0.1,
    population_size=80,
    n_generations=50,
)
ga_mask = np.array(ga_best, dtype=bool)

# Evaluate GA result with 5-fold CV
clf_ga = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
ga_scores = cross_val_score(clf_ga, X[:, ga_mask], y, cv=cv5, scoring='accuracy')

print(f"\nSingle-objective GA result:")
print(f"  Features selected: {ga_mask.sum()}/{N_FEATURES}")
print(f"  Accuracy (5-fold CV): {ga_scores.mean():.4f} ± {ga_scores.std():.4f}")

In [ ]:
# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Pareto front with GA single result ---
ax = axes[0]
errors_pf = objectives[:, 0]
features_pf = objectives[:, 1].astype(int)

ax.scatter(features_pf, errors_pf, c='steelblue', s=60, zorder=5,
           label='NSGA-II Pareto front')
ax.plot(features_pf, errors_pf, 'b--', alpha=0.4)
ax.scatter(features_pf[knee_idx], errors_pf[knee_idx],
           c='red', s=200, zorder=7, marker='*', label='NSGA-II knee point')

# Single-objective GA point
ax.scatter(
    ga_mask.sum(),
    1.0 - ga_scores.mean(),
    c='orange', s=200, zorder=7, marker='D',
    label=f'Single-obj GA (α=0.1)'
)

ax.axhline(BASELINE_ERROR, color='gray', linestyle=':', linewidth=1.5,
           label=f'Baseline (all {N_FEATURES} features)')

ax.set_xlabel('Number of features selected', fontsize=12)
ax.set_ylabel('Cross-validation error rate', fontsize=12)
ax.set_title('NSGA-II vs Single-Objective GA', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Right: Method comparison ---
ax2 = axes[1]
methods = ['Baseline\n(30 feat)', 'Single-obj GA\n(α=0.1)', 'NSGA-II\nKnee']
accs = [
    1 - BASELINE_ERROR,
    ga_scores.mean(),
    1 - knee_error,
]
n_feats = [
    N_FEATURES,
    ga_mask.sum(),
    knee_features,
]
colours = ['#888888', '#FFA500', '#DC143C']
bars = ax2.bar(methods, accs, color=colours, alpha=0.8, edgecolor='black')

# Annotate with feature counts
for bar, n, acc in zip(bars, n_feats, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.4f}\n({n} feats)', ha='center', va='bottom', fontsize=10)

ax2.set_ylim(0.90, 1.0)
ax2.set_ylabel('5-fold CV accuracy', fontsize=12)
ax2.set_title('Method Comparison', fontsize=13)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('nsga2_vs_ga_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nSummary:")
print(f"  Baseline:          {1-BASELINE_ERROR:.4f} accuracy, {N_FEATURES} features")
print(f"  Single-obj GA:     {ga_scores.mean():.4f} accuracy, {ga_mask.sum()} features")
print(f"  NSGA-II (knee):    {1-knee_error:.4f} accuracy, {knee_features} features")

## 10. Key Takeaways

1. **NSGA-II returns a trade-off surface (Pareto front)**, not a single solution. The decision about which trade-off to accept is deferred to a human or domain constraint.

2. **The knee point is a principled automatic selection strategy**: it identifies the point of diminishing returns on the Pareto front — where adding more features yields little accuracy improvement.

3. **The single-objective GA commits to a trade-off upfront** via the penalty coefficient $\alpha$. Different $\alpha$ values produce different feature counts. NSGA-II explores all trade-offs in one run.

4. **DEAP makes NSGA-II straightforward to implement** with `creator`, `toolbox`, `tools.selNSGA2`, and `tools.selTournamentDCD`.

5. **Caching fitness evaluations** is essential for performance: with 80 individuals × 50 generations, the naive approach requires 4,000 cross-validation runs. Caching reduces this to the number of unique feature subsets evaluated.

---

**References**:
- Deb et al. (2002). NSGA-II. *IEEE TEC* 6(2), 182–197.
- DEAP: https://deap.readthedocs.io
- pymoo (NSGA-III, MOEA/D): https://pymoo.org